# Практическая работа №4: Элементы корреляционного анализа. Проверка статистической гипотезы о равенстве коэффициента корреляции нулю

Выполнили студенты гр. 2384 Федоров Михаил и Муравин Егор. Вариант №30

## Цель работы

Освоение основных понятий, связанных с корреляционной зависимостью между случайными величинами, статистическими гипотезами и проверкой их «справедливости».

## Постановка задачи

Из заданной генеральной совокупности сформировать выборку по второму признаку. Провести статистическую обработку второй выборки в объёме практических работ №1 и №2, с целью определения точечных статистических оценок параметров распределения исследуемого признака (математического ожидания, дисперсии, среднеквадратичного отклонения, асимметрии, эксцесса и коэффициента вариации). Для системы двух случайных величин X (первый признак) и Y (второй признак) сформировать двумерную выборку и найти статистическую оценку коэффициента корреляции, построить доверительный интервал для коэффициента корреляции и осуществить проверку статистической гипотезы о равенстве коэффициента корреляции нулю. Полученные результаты содержательно проинтерпретировать.

## Основные теоретические положения

#### Стандартная формула по корреляционной таблице

Средние значения признаков:

$$\bar{x} = \frac{1}{N}\sum_{j=1}^{k_x} n_{x_j} \cdot x_j, \quad \bar{y} = \frac{1}{N}\sum_{i=1}^{k_y} n_{y_i} \cdot y_i$$

где $x_j$, $y_i$ - середины интервалов, $N$ - объём выборки.

Коэффициент корреляции:

$$r_{xy} = \frac{\sum\limits_{i=1}^{k_y}\sum\limits_{j=1}^{k_x} n_{ij}(x_j - \bar{x})(y_i - \bar{y})}{\sqrt{\sum\limits_{j=1}^{k_x} n_{x_j}(x_j - \bar{x})^2 \cdot \sum\limits_{i=1}^{k_y} n_{y_i}(y_i - \bar{y})^2}}$$

#### Формула через условные варианты

Условные варианты:

$$u_j = \frac{x_j - C_x}{h_x}, \quad v_i = \frac{y_i - C_y}{h_y}$$

где $C_x$, $C_y$ - ложные нули (середины модальных интервалов), $h_x$, $h_y$ - шаги интервалов.

Условные моменты:

$$M_{10} = \frac{1}{N}\sum_{j=1}^{k_x} n_{x_j} \cdot u_j, \quad M_{01} = \frac{1}{N}\sum_{i=1}^{k_y} n_{y_i} \cdot v_i$$

$$M_{20} = \frac{1}{N}\sum_{j=1}^{k_x} n_{x_j} \cdot u_j^2, \quad M_{02} = \frac{1}{N}\sum_{i=1}^{k_y} n_{y_i} \cdot v_i^2$$

$$M_{11} = \frac{1}{N}\sum_{i=1}^{k_y}\sum_{j=1}^{k_x} n_{ij} \cdot u_j \cdot v_i$$

Коэффициент корреляции:

$$r_{xy} = \frac{M_{11} - M_{10} \cdot M_{01}}{\sqrt{(M_{20} - M_{10}^2)(M_{02} - M_{01}^2)}}$$

### Доверительный интервал для коэффициента корреляции

z-преобразование Фишера:

$$z = \frac{1}{2} \ln \frac{1+r}{1-r} $$

Стандартная ошибка $z$-статистики:

$$\sigma_z = \frac{1}{\sqrt{n-3}}$$

Доверительный интервал для $z$ с надёжностью $\gamma$:

$$z - z_{\gamma} \cdot \sigma_z < \zeta < z + z_{\gamma} \cdot \sigma_z$$

где $z_{\gamma}$ - квантиль стандартного нормального распределения уровня $\frac{1+\gamma}{2}$.

Границы доверительного интервала для коэффициента корреляции $\rho$ получаются обратным преобразованием:

$$r_{\text{нижн}} = \tanh(z_{\text{нижн}}), \quad r_{\text{верхн}} = \tanh(z_{\text{верхн}})$$

### Проверка гипотезы о равенстве коэффициента корреляции нулю


Статистика критерия:

$$t_{\text{набл}} = \frac{r\sqrt{n-2}}{\sqrt{1-r^2}}$$

которая при справедливости $H_0$ имеет распределение Стьюдента с $k = n - 2$ степенями свободы.

Правило принятия решения:

- Если $|t_{\text{набл}}| > t_{\text{крит}}(\alpha, n-2)$ - гипотеза $H_0$ отвергается, корреляция статистически значима.
- Если $|t_{\text{набл}}| \leq t_{\text{крит}}(\alpha, n-2)$ - нет оснований отвергнуть $H_0$.

## Выполнение работы


1. Провести статистическую обработку второй выборки в объёме практических работ №1 и №2, с целью определения точечных статистических оценок параметров распределения исследуемого признака (математического ожидания, дисперсии, среднеквадратичного отклонения, асимметрии, эксцесса, моды, медианы и коэффициента вариации). Оформить результаты в виде таблицы, сделать выводы.

In [47]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
from scipy.stats import norm, chi2, chi, t as t_dist
from scipy.optimize import brentq

In [48]:
df = pd.read_csv("data_1.csv", sep=",")
df.rename(columns={"nu": "relative_weight", "E": "simplicity"}, inplace=True)
df = df.sample(n=118, random_state=1)
df

,relative_weight,simplicity
94,458,133.5
54,440,126.7
59,437,129.2
115,429,112.9
74,503,149.9
...,...,...
9,566,175.7
72,436,114.3
12,500,155.5
107,471,119.7


In [49]:
def build_intervals_row(data):
    x_min = data.min()
    x_max = data.max()
    n = len(data)
    k = math.ceil(1 + 3.322 * math.log10(n))
    h = (x_max - x_min) / k

    boundaries = [x_min]
    for _ in range(k):
        boundaries.append(round(boundaries[-1] + h, 2))
    if boundaries[-1] < x_max:
        boundaries.append(round(boundaries[-1] + h, 2))

    counts, edges = np.histogram(data, bins=boundaries)
    midpoints = (edges[:-1] + edges[1:]) / 2

    return midpoints, counts, edges, h, n, k

In [50]:
def compute_all_statistics(data):
    # Интервальный ряд
    midpoints, counts, edges, h, n, k = build_intervals_row(data)

    # Выборочное среднее
    x_bar = np.average(midpoints, weights=counts)

    # Ложный нуль
    modal_idx = np.argmax(counts)
    C = midpoints[modal_idx]

    # Условные варианты
    u = (midpoints - C) / h

    # Условные моменты
    M1_star = np.sum(counts * u) / n
    M2_star = np.sum(counts * u**2) / n
    M3_star = np.sum(counts * u**3) / n
    M4_star = np.sum(counts * u**4) / n

    # Центральные моменты
    m1 = 0
    m2 = (M2_star - M1_star**2) * h**2
    m3 = (M3_star - 3 * M2_star * M1_star + 2 * M1_star**3) * h**3
    m4 = (M4_star - 4 * M3_star * M1_star + 6 * M2_star * M1_star**2 - 3 * M1_star**4) * h**4

    # Дисперсия и СКО
    D = np.average((midpoints - x_bar)**2, weights=counts)
    sigma = np.sqrt(D)
    s2 = D * n / (n - 1)
    s = np.sqrt(s2)

    # Асимметрия и эксцесс
    A_s = m3 / sigma**3
    E_k = m4 / sigma**4 - 3

    # Мода
    f_m = counts[modal_idx]
    f_m_minus = counts[modal_idx - 1] if modal_idx > 0 else 0
    f_m_plus = counts[modal_idx + 1] if modal_idx < len(counts) - 1 else 0
    x_m = edges[modal_idx]
    denom = (f_m - f_m_minus) + (f_m - f_m_plus)
    Mo = x_m + h * (f_m - f_m_minus) / denom if denom != 0 else x_m

    # Медиана
    cumulative = 0
    me_idx = 0
    for i, c in enumerate(counts):
        cumulative += c
        if cumulative >= n / 2:
            me_idx = i
            break
    S_me_minus = cumulative - counts[me_idx]
    Me = edges[me_idx] + h * (n / 2 - S_me_minus) / counts[me_idx]

    # Коэффициент вариации
    V = (sigma / x_bar) * 100

    results = {
        'n': n,
        'k': k,
        'h': round(h, 4),
        'x_bar': round(x_bar, 4),
        'D': round(D, 4),
        's2': round(s2, 4),
        'sigma': round(sigma, 4),
        's': round(s, 4),
        'A_s': round(A_s, 4),
        'E_k': round(E_k, 4),
        'Mo': round(Mo, 4),
        'Me': round(Me, 4),
        'V': round(V, 2),
        'midpoints': midpoints,
        'counts': counts,
        'edges': edges,
        'C': C,
    }

    return results

In [51]:
statistics_rw = compute_all_statistics(df["relative_weight"])
statistics_s = compute_all_statistics(df["simplicity"])

In [52]:
summary_table = pd.DataFrame({
    "Параметр": [
        "Объём выборки n",
        "Число интервалов k",
        "Шаг интервала h",
        "Выборочное среднее x̄",
        "Смещённая дисперсия D",
        "Исправленная дисперсия s²",
        "Смещённое СКО σ",
        "Исправленное СКО s",
        "Коэффициент асимметрии Aₛ",
        "Коэффициент эксцесса E",
        "Мода Mo",
        "Медиана Me",
        "Коэффициент вариации V, %"
    ],
    "relative_weight": [
        statistics_rw['n'], statistics_rw['k'], statistics_rw['h'],
        statistics_rw['x_bar'], statistics_rw['D'], statistics_rw['s2'],
        statistics_rw['sigma'], statistics_rw['s'],
        statistics_rw['A_s'], statistics_rw['E_k'],
        statistics_rw['Mo'], statistics_rw['Me'], statistics_rw['V']
    ],
    "simplicity": [
        statistics_s['n'], statistics_s['k'], statistics_s['h'],
        statistics_s['x_bar'], statistics_s['D'], statistics_s['s2'],
        statistics_s['sigma'], statistics_s['s'],
        statistics_s['A_s'], statistics_s['E_k'],
        statistics_s['Mo'], statistics_s['Me'], statistics_s['V']
    ]
})

summary_table

,Параметр,relative_weight,simplicity
0,Объём выборки n,118.0000,118.0000
1,Число интервалов k,8.0000,8.0000
2,Шаг интервала h,37.8750,15.5750
3,Выборочное среднее x̄,458.0373,129.7188
4,Смещённая дисперсия D,2803.5264,511.6526
5,Исправленная дисперсия s²,2827.4882,516.0257
6,Смещённое СКО σ,52.9483,22.6197
7,Исправленное СКО s,53.1741,22.7162
8,Коэффициент асимметрии Aₛ,0.3569,0.2339
9,Коэффициент эксцесса E,-0.0031,0.1865


2. Построить двумерный интервальный вариационный ряд, оформить в виде таблицы.

In [53]:
rw = df["relative_weight"].values
s = df["simplicity"].values

edges_rw = statistics_rw['edges']
edges_s = statistics_s['edges']


# Построение двумерного интервального ряда
k_x = len(edges_rw) - 1
k_y = len(edges_s) - 1

freq_matrix = np.zeros((k_y, k_x), dtype=int)

for xi, yi in zip(rw, s):
    ix = np.searchsorted(edges_rw, xi, side='right') - 1
    ix = min(ix, k_x - 1)
    ix = max(ix, 0)

    iy = np.searchsorted(edges_s, yi, side='right') - 1
    iy = min(iy, k_y - 1)
    iy = max(iy, 0)

    freq_matrix[iy, ix] += 1

x_labels = [f"[{edges_rw[i]:.2f}, {edges_rw[i+1]:.2f})" for i in range(k_x)]
y_labels = [f"[{edges_s[i]:.2f}, {edges_s[i+1]:.2f})" for i in range(k_y)]

table = pd.DataFrame(freq_matrix, index=y_labels, columns=x_labels)
table.index.name = "simplicity"        
table.columns.name = "relative_weight" 
table

relative_weight,"[320.00, 357.88)","[357.88, 395.76)","[395.76, 433.64)","[433.64, 471.52)","[471.52, 509.40)","[509.40, 547.28)","[547.28, 585.16)","[585.16, 623.04)"
simplicity,,,,,,,,
"[71.10, 86.68)",2,2,0,0,0,0,0,0
"[86.68, 102.26)",0,4,2,0,0,0,0,0
"[102.26, 117.84)",0,4,14,4,2,0,0,0
"[117.84, 133.42)",0,0,11,25,2,0,0,0
"[133.42, 148.99)",0,0,0,9,12,3,0,0
"[148.99, 164.56)",0,0,0,0,4,9,1,0
"[164.56, 180.14)",0,0,0,0,0,2,3,0
"[180.14, 195.71)",0,0,0,0,0,1,0,2


3. По полученному двумерному интервальному вариационному ряду построить корреляционную таблицу, сделать выводы.

In [54]:
midpoints_rw = statistics_rw['midpoints']
midpoints_s = statistics_s['midpoints']

corr_table = pd.DataFrame(freq_matrix, index=[f"{m:.2f}" for m in midpoints_s], columns=[f"{m:.2f}" for m in midpoints_rw])

corr_table["n_y"] = corr_table.sum(axis=1)
corr_table.loc["n_x"] = corr_table.sum(axis=0)
corr_table.index.name = "simplicity"        
corr_table.columns.name = "relative_weight" 

corr_table

relative_weight,338.94,376.82,414.70,452.58,490.46,528.34,566.22,604.10,n_y
simplicity,,,,,,,,,
78.89,2,2,0,0,0,0,0,0,4
94.47,0,4,2,0,0,0,0,0,6
110.05,0,4,14,4,2,0,0,0,24
125.63,0,0,11,25,2,0,0,0,38
141.20,0,0,0,9,12,3,0,0,24
156.78,0,0,0,0,4,9,1,0,14
172.35,0,0,0,0,0,2,3,0,5
187.93,0,0,0,0,0,1,0,2,3
n_x,2,10,27,38,20,15,4,2,118


### Вывод
Из корреляционной таблицы видно, что частоты распределены вдоль диагонали таблицы, что свидетельствует о наличии положительной корреляционной связи между признаками X и Y. 

4. Исходя из результатов корреляционной таблицы вычислить значение выборочного коэффициента корреляции двумя способами: с помощью стандартной формулы и с помощью условных вариант. Убедиться, что результаты совпадают. Сделать выводы.

## Способ 1: Стандартная формула по исходным данным


Маргинальные частоты:

$$n_{x_j} = \sum_{i=1}^{k_y} n_{ij}, \quad n_{y_i} = \sum_{j=1}^{k_x} n_{ij}$$

Средние значения:

$$\bar{x} = \frac{1}{N}\sum_{j=1}^{k_x} n_{x_j} \cdot x_j, \quad \bar{y} = \frac{1}{N}\sum_{i=1}^{k_y} n_{y_i} \cdot y_i$$


Коэффициент корреляции:

$$r_{xy} = \frac{\sum\limits_{i=1}^{k_y}\sum\limits_{j=1}^{k_x} n_{ij}(x_j - \bar{x})(y_i - \bar{y})}{\sqrt{\sum\limits_{j=1}^{k_x} n_{x_j}(x_j - \bar{x})^2 \cdot \sum\limits_{i=1}^{k_y} n_{y_i}(y_i - \bar{y})^2}}$$




In [55]:
# Маргинальные частоты
n_x_marg = np.sum(freq_matrix, axis=0)  
n_y_marg = np.sum(freq_matrix, axis=1) 

# Средние по серединам интервалов
x_mean = np.average(midpoints_rw, weights=n_x_marg)   
y_mean = np.average(midpoints_s, weights=n_y_marg) 

# Ковариация: Σ n_ij * (x_j - x̄) * (y_i - ȳ)
cov_xy = 0
for iy in range(k_y):         
    for ix in range(k_x):   
        cov_xy += freq_matrix[iy, ix] * (midpoints_rw[ix] - x_mean) * (midpoints_s[iy] - y_mean)

# Дисперсии
var_x = np.sum(n_x_marg * (midpoints_rw - x_mean)**2)
var_y = np.sum(n_y_marg * (midpoints_s - y_mean)**2)

denom = np.sqrt(var_x * var_y)
r_grouped = cov_xy / denom if denom != 0 else 0
print(f"r (по корреляционной таблице) = {r_grouped:.6f}")

r (по корреляционной таблице) = 0.887289


## Способ 2: С помощью условных вариант по корреляционной таблице

Условные варианты:

$$u_j = \frac{x_j - C_x}{h_x}, \quad v_i = \frac{y_i - C_y}{h_y}$$


Условные моменты:

$$M_{10} = \frac{1}{N}\sum_{j=1}^{k_x} n_{x_j} \cdot u_j, \quad M_{01} = \frac{1}{N}\sum_{i=1}^{k_y} n_{y_i} \cdot v_i$$

$$M_{20} = \frac{1}{N}\sum_{j=1}^{k_x} n_{x_j} \cdot u_j^2, \quad M_{02} = \frac{1}{N}\sum_{i=1}^{k_y} n_{y_i} \cdot v_i^2$$

$$M_{11} = \frac{1}{N}\sum_{i=1}^{k_y}\sum_{j=1}^{k_x} n_{ij} \cdot u_j \cdot v_i$$

Коэффициент корреляции:

$$r_{xy} = \frac{M_{11} - M_{10} \cdot M_{01}}{\sqrt{(M_{20} - M_{10}^2)(M_{02} - M_{01}^2)}}$$

In [56]:
C_x = midpoints_rw[np.argmax(n_x_marg)]
C_y = midpoints_s[np.argmax(n_y_marg)]

u = (midpoints_rw - C_x) / statistics_rw['h']
v = (midpoints_s - C_y) / statistics_s['h']

M10 = np.sum(n_x_marg * u) / statistics_rw['n']
M01 = np.sum(n_y_marg * v) / statistics_rw['n']
M20 = np.sum(n_x_marg * u**2) / statistics_rw['n']
M02 = np.sum(n_y_marg * v**2) / statistics_rw['n']

# M11 = (1/n) Σ n_ij u_i v_j
M11 = 0
for i in range(len(u)):
    for j in range(len(v)):
        M11 += freq_matrix[i, j] * u[i] * v[j]
M11 /= statistics_rw['n']

r_cond = (M11 - M10 * M01) / np.sqrt((M20 - M10**2) * (M02 - M01**2))
print(f"r (по корреляционной таблице через условные варианты) = {r_cond:.6f}")


r (по корреляционной таблице через условные варианты) = 0.887289


In [57]:
print(f"Сравнение результатов:")
print(f"r (стандартная формула) = {r_grouped:.6f}")
print(f"r (условные варианты)  = {r_cond:.6f}")
print(f"Разница = {abs(r_grouped - r_cond):.8f}")

Сравнение результатов:
r (стандартная формула) = 0.887289
r (условные варианты)  = 0.887289
Разница = 0.00000000


### Вывод
Коэффициент корреляции вычисленн двумя способами. Результаты оказались идентичны друг к другу. Небольшое расхождение могут быть обусловлено потерей точности при группировке данных в интервальный ряд. Полученное значение коэффициента корреляции свидетельствует о наличии сильной положительной линейной связи между признаками, вследствие чего увеличение одного параметра влечет за собой возрастание другого параметра.


In [58]:
def fisher_z(r):
    return 0.5 * np.log((1 + r) / (1 - r))

def inv_fisher_z(z):
    return (np.exp(2*z) - 1) / (np.exp(2*z) + 1)

def confidence_interval_correlation(r, n, gamma):
    n = statistics_rw['n']
    z = fisher_z(r)
    se_z = 1 / np.sqrt(n - 3)

    z_crit = stats.norm.ppf((1 + gamma) / 2)
    z_lower = z - z_crit * se_z
    z_upper = z + z_crit * se_z
    r_lower = inv_fisher_z(z_lower)
    r_upper = inv_fisher_z(z_upper)
    return r_lower, r_upper, z, se_z, z_crit


print("Доверительные интервалы для коэффициента корреляции:")
print(f"r = {r_grouped:.6f}, n = {statistics_rw['n']}")
print()

for gamma in [0.95, 0.99]:
    r_lo, r_hi, z_val, sig_z, z_cr = confidence_interval_correlation(r_cond, statistics_rw['n'], gamma)
    print(f"γ = {gamma}:")
    print(f"z-преобразование Фишера: z = {z_val:.6f}")
    print(f"Стандартная ошибка σ_z = {sig_z:.6f}")
    print(f"Критическое значение z_крит = {z_cr:.4f}")
    print(f"Доверительный интервал: ({r_lo:.6f}, {r_hi:.6f})")
    print()


Доверительные интервалы для коэффициента корреляции:
r = 0.887289, n = 118

γ = 0.95:
z-преобразование Фишера: z = 1.409037
Стандартная ошибка σ_z = 0.093250
Критическое значение z_крит = 1.9600
Доверительный интервал: (0.841494, 0.920426)

γ = 0.99:
z-преобразование Фишера: z = 1.409037
Стандартная ошибка σ_z = 0.093250
Критическое значение z_крит = 2.5758
Доверительный интервал: (0.823900, 0.928753)



### Вывод
При увеличении надёжности γ доверительный интервал расширяется. Ширина интервала зависит от объёма выборки: чем больше n, тем уже интервал и точнее оценка. В обоих случаях границы интервалов весьма велики, что свидетельствует о сильной кореляции между данными.

6. Осуществить проверку статистической гипотезы о равенстве коэффициента корреляции нулю при заданном уровне значимости $\alpha = 0.05$, сделать выводы.

In [59]:
n = statistics_rw['n']
alpha = 0.05
t_stat = r_grouped * np.sqrt((n - 2) / (1 - r_grouped**2))
t_crit = stats.t.ppf(1 - alpha/2, df=n-2)
p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=n-2))

print(f"t-статистика = {t_stat:.6f}")
print(f"Критическое значение t ({n-2} степеней свободы, α=0.05) = {t_crit:.6f}")
print(f"p-значение = {p_value:.6f}")

t-статистика = 20.720149
Критическое значение t (116 степеней свободы, α=0.05) = 1.980626
p-значение = 0.000000


### Вывод
Проверка гипотезы о равенстве коэффициента корреляции нулю показала, что наблюдаемое значение t-статистики превышает критическое значение при заданном уровне значимости. Полученное p-значение оказывается значительно меньше уровня значимости, что свидетельствует о крайне низкой вероятности ошибочно отвергнуть нулевую гипотезу. Вследствие этого нулевая гипотеза отвергается, между признаками существует линейная корреляционная связь.

## Выводы
В ходе выполнения практической работы была проведена статистическая обработка второго признака по аналогии с вычислениями из прошлых практических работ. Был построен двумерный интервальный вариационный ряд и корреляционная таблица, на основе которой двумя способами (стандартная формула и метод условных вариант) вычислен выборочный коэффициент корреляции. Его высокое значение, около 0,87, свидетельствует о сильной положительной линейной связи между признаками. Также построены доверительные интервалы. Проведена проверка статистической гипотезы о равенстве коэффициента корреляции нулю при уровне значимости 0,05. В результате нулевая гипотеза была отвергнута, что также подтвердило наличие статистически значимой корреляционной зависимости между исследуемыми величинами.
